# Working with Technical Indicators

This tutorial covers how to use and combine technical indicators in Backtrader.

## What You'll Learn

1. Using built-in indicators
2. Combining multiple indicators
3. Creating custom indicators
4. Indicator parameters and optimization
5. Plotting indicators

In [ ]:
import backtrader as bt
import datetime
import pandas as pd
import numpy as np

# Create sample data
dates = pd.date_range('2023-01-01', periods=200, freq='D')
prices = 100 + np.cumsum(np.random.randn(200) * 2)

data = pd.DataFrame({
    'open': prices + np.random.randn(200) * 0.5,
    'high': prices + abs(np.random.randn(200) * 1),
    'low': prices - abs(np.random.randn(200) * 1),
    'close': prices,
    'volume': np.random.randint(1000, 10000, 200)
}, index=dates)

data.to_csv('sample_data_indicators.csv')
print("Sample data created")

## 1. Using Built-in Indicators

Backtrader includes 60+ technical indicators. Let's explore the most common ones.

In [ ]:
class IndicatorDemo(bt.Strategy):
    """Strategy demonstrating various indicators."""
    
    def __init__(self):
        # Moving Averages
        self.sma = bt.indicators.SMA(self.data.close, period=20)
        self.ema = bt.indicators.EMA(self.data.close, period=20)
        
        # Momentum Indicators
        self.rsi = bt.indicators.RSI(self.data.close, period=14)
        self.macd = bt.indicators.MACD(self.data.close)
        
        # Volatility Indicators
        self.bbands = bt.indicators.BollingerBands(self.data.close, period=20)
        self.atr = bt.indicators.ATR(self.data, period=14)
        
        # Volume Indicators
        self.volume_sma = bt.indicators.SMA(self.data.volume, period=20)
    
    def next(self):
        # Access indicator values
        print(f'Date: {self.data.datetime.date(0)}')
        print(f'  SMA: {self.sma[0]:.2f}')
        print(f'  RSI: {self.rsi[0]:.2f}')
        print(f'  MACD: {self.macd.macd[0]:.2f}')
        print(f'  BB Upper: {self.bbands.top[0]:.2f}')
        print(f'  ATR: {self.atr[0]:.2f}')

# Run the strategy
cerebro = bt.Cerebro()
data_feed = bt.feeds.GenericCSVData(
    dataname='sample_data_indicators.csv',
    datetime=0, open=1, high=2, low=3, close=4, volume=5,
    dtformat='%Y-%m-%d'
)
cerebro.adddata(data_feed)
cerebro.addstrategy(IndicatorDemo)
cerebro.run()

## 2. Combining Multiple Indicators

Create a strategy that uses multiple indicators together.

In [ ]:
class MultiIndicatorStrategy(bt.Strategy):
    """Strategy using multiple indicators for entry/exit signals."""
    
    params = (
        ('rsi_low', 30),
        ('rsi_high', 70),
        ('macd_fast', 12),
        ('macd_slow', 26),
    )
    
    def __init__(self):
        # Trend indicators
        self.sma_fast = bt.indicators.SMA(period=10)
        self.sma_slow = bt.indicators.SMA(period=30)
        self.trend = self.sma_fast > self.sma_slow
        
        # Momentum indicators
        self.rsi = bt.indicators.RSI(period=14)
        self.macd = bt.indicators.MACD(
            period_me1=self.p.macd_fast,
            period_me2=self.p.macd_slow
        )
        
        # Volatility filter
        self.atr = bt.indicators.ATR(period=14)
        self.atr_sma = bt.indicators.SMA(self.atr, period=20)
        
    def next(self):
        # Entry conditions
        if not self.position:
            # Buy when:
            # 1. Uptrend (fast MA > slow MA)
            # 2. RSI oversold
            # 3. MACD bullish crossover
            # 4. Normal volatility
            if (self.trend[0] and 
                self.rsi[0] < self.p.rsi_low and
                self.macd.macd[0] > self.macd.signal[0] and
                self.atr[0] < self.atr_sma[0] * 1.5):
                self.buy()
        
        # Exit conditions
        else:
            # Sell when:
            # 1. RSI overbought OR
            # 2. MACD bearish crossover OR
            # 3. Trend reversal
            if (self.rsi[0] > self.p.rsi_high or
                self.macd.macd[0] < self.macd.signal[0] or
                not self.trend[0]):
                self.close()

# Run backtest
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(MultiIndicatorStrategy)
cerebro.broker.setcash(10000.0)
cerebro.broker.setcommission(commission=0.001)

print(f'Starting Value: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 3. Creating Custom Indicators

Build your own indicators by extending the Indicator class.

In [ ]:
class CustomMomentum(bt.Indicator):
    """Custom momentum indicator.
    
    Calculates price momentum over a period.
    """
    lines = ('momentum',)
    params = (('period', 14),)
    
    def __init__(self):
        # Momentum = Current Price - Price N periods ago
        self.lines.momentum = self.data.close - self.data.close(-self.p.period)


class CustomVolatilityRatio(bt.Indicator):
    """Custom volatility ratio indicator.
    
    Compares recent volatility to historical average.
    """
    lines = ('vol_ratio',)
    params = (
        ('period', 20),
        ('lookback', 100),
    )
    
    def __init__(self):
        # Recent volatility
        recent_std = bt.indicators.StdDev(self.data.close, period=self.p.period)
        
        # Historical average volatility
        hist_std = bt.indicators.SMA(recent_std, period=self.p.lookback)
        
        # Ratio
        self.lines.vol_ratio = recent_std / hist_std


class CustomIndicatorStrategy(bt.Strategy):
    """Strategy using custom indicators."""
    
    def __init__(self):
        self.momentum = CustomMomentum(self.data, period=10)
        self.vol_ratio = CustomVolatilityRatio(self.data)
        
    def next(self):
        if not self.position:
            # Buy on positive momentum with low volatility
            if self.momentum[0] > 0 and self.vol_ratio[0] < 1.2:
                self.buy()
        else:
            # Sell on negative momentum or high volatility
            if self.momentum[0] < 0 or self.vol_ratio[0] > 1.5:
                self.close()

# Test custom indicators
cerebro = bt.Cerebro()
cerebro.adddata(data_feed)
cerebro.addstrategy(CustomIndicatorStrategy)
cerebro.broker.setcash(10000.0)
results = cerebro.run()
print(f'Final Value: {cerebro.broker.getvalue():.2f}')

## 4. Indicator Parameters and Optimization

Find optimal indicator parameters using Cerebro's optimization feature.

In [ ]:
class OptimizableStrategy(bt.Strategy):
    """Strategy with optimizable parameters."""
    
    params = (
        ('sma_period', 20),
        ('rsi_period', 14),
        ('rsi_low', 30),
        ('rsi_high', 70),
    )
    
    def __init__(self):
        self.sma = bt.indicators.SMA(period=self.p.sma_period)
        self.rsi = bt.indicators.RSI(period=self.p.rsi_period)
    
    def next(self):
        if not self.position:
            if (self.data.close[0] > self.sma[0] and 
                self.rsi[0] < self.p.rsi_low):
                self.buy()
        else:
            if self.rsi[0] > self.p.rsi_high:
                self.close()

# Optimize parameters
cerebro = bt.Cerebro(optreturn=False)
cerebro.adddata(data_feed)

# Add strategy with parameter ranges
cerebro.optstrategy(
    OptimizableStrategy,
    sma_period=range(10, 31, 5),  # 10, 15, 20, 25, 30
    rsi_period=range(10, 21, 2),  # 10, 12, 14, 16, 18, 20
)

cerebro.broker.setcash(10000.0)
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')

print("Running optimization...")
opt_results = cerebro.run()

# Find best parameters
best_sharpe = -999
best_params = None

for result in opt_results:
    strat = result[0]
    sharpe = strat.analyzers.sharpe.get_analysis().get('sharperatio', 0)
    if sharpe and sharpe > best_sharpe:
        best_sharpe = sharpe
        best_params = {
            'sma_period': strat.p.sma_period,
            'rsi_period': strat.p.rsi_period,
        }

print(f"\nBest Parameters: {best_params}")
print(f"Best Sharpe Ratio: {best_sharpe:.3f}")

## 5. Common Indicator Patterns

### Moving Average Crossovers

In [ ]:
class MACrossStrategy(bt.Strategy):
    """Golden Cross / Death Cross strategy."""
    
    def __init__(self):
        sma_fast = bt.indicators.SMA(period=50)
        sma_slow = bt.indicators.SMA(period=200)
        self.crossover = bt.indicators.CrossOver(sma_fast, sma_slow)
    
    def next(self):
        if self.crossover > 0:  # Golden cross
            if not self.position:
                self.buy()
        elif self.crossover < 0:  # Death cross
            if self.position:
                self.close()

### RSI Divergence

In [ ]:
class RSIDivergence(bt.Indicator):
    """Detect RSI divergence."""
    lines = ('divergence',)
    params = (('period', 14), ('lookback', 5))
    
    def __init__(self):
        self.rsi = bt.indicators.RSI(period=self.p.period)
    
    def next(self):
        # Bullish divergence: price makes lower low, RSI makes higher low
        if (self.data.close[0] < self.data.close[-self.p.lookback] and
            self.rsi[0] > self.rsi[-self.p.lookback]):
            self.lines.divergence[0] = 1
        # Bearish divergence: price makes higher high, RSI makes lower high
        elif (self.data.close[0] > self.data.close[-self.p.lookback] and
              self.rsi[0] < self.rsi[-self.p.lookback]):
            self.lines.divergence[0] = -1
        else:
            self.lines.divergence[0] = 0

## Summary

You've learned:
- ✅ How to use 60+ built-in indicators
- ✅ Combining multiple indicators for better signals
- ✅ Creating custom indicators
- ✅ Optimizing indicator parameters
- ✅ Common indicator patterns

## Next Steps

- **Tutorial 03**: Position Sizing and Risk Management
- **Tutorial 04**: Parameter Optimization Techniques
- **Tutorial 05**: Live Trading with CCXT

## Resources

- [Indicator API Reference](/api/indicator.md)
- [Built-in Indicators List](../user_guide/indicators.md)
- [Custom Indicator Guide](/advanced/custom-indicators.md)